In [32]:
# ============================================================
# QUESTION 4 - SETUP
# ============================================================

import nltk
import random
import time
import math
import os
import pickle
import statistics
from collections import Counter, defaultdict

import pandas as pd

from nltk.corpus import treebank, gutenberg, brown
from nltk import induce_pcfg
from nltk import Tree

# Required corpora
nltk.download("treebank")
nltk.download("gutenberg")
nltk.download("punkt")
nltk.download("punkt_tab")

print("Q4 setup complete.")

Q4 setup complete.


[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [33]:
from tqdm.notebook import tqdm

In [34]:
# ============================================================
# LOAD Q1 + Q3 TRAINED COMPONENTS
# ============================================================

from google.colab import drive
import os
import cloudpickle

drive.mount("/content/drive")

MODEL_DIR = "/content/drive/MyDrive/GROUP_ASSIGNMENT_1/models"

Q1_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "q1_models.pkl"
)

Q3_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "q3_models.pkl"
)

# ------------------------------------------------------------
# Load Q1
# ------------------------------------------------------------

with open(Q1_MODEL_PATH, "rb") as f:
    q1_models = cloudpickle.load(f)

lm_en = q1_models["lm_en"]
hmm_en = q1_models["hmm_en"]
vocab_en = q1_models["vocab_en"]

# ------------------------------------------------------------
# Load Q3
# ------------------------------------------------------------

with open(Q3_MODEL_PATH, "rb") as f:
    q3_models = cloudpickle.load(f)

vocab = q3_models["vocab"]
unigram_freq = q3_models["unigram_freq"]
bigram_freq = q3_models["bigram_freq"]
unigram_ctx_freq = q3_models["unigram_ctx_freq"]
symspell_index = q3_models["symspell_index"]

correct_nonword = q3_models["correct_nonword"]
correct_realword = q3_models["correct_realword"]

print("✓ Q1 models loaded")
print("✓ Q3 models loaded")
print("✓ Q4 will reuse Q1 and Q3 components")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Q1 models loaded
✓ Q3 models loaded
✓ Q4 will reuse Q1 and Q3 components


In [35]:
# ============================================================
# Q4 CONFIGURATION
# ============================================================

MERGE_PROBABILITY = 0.08
GRAMMAR_TRIGGER_N = 8
TYPO_PROBABILITY = 0.10   # simulated mistyped words (spelling test)

# Q1's already-selected settings
Q1_MAX_WORD_LEN = lm_en.max_word_len
Q1_BEAM_SIZE = 25

print("Q4 configuration")
print("-----------------")
print(f"Merge probability p       = {MERGE_PROBABILITY}")
print(f"Grammar trigger interval N = {GRAMMAR_TRIGGER_N}")
print(f"Typo probability          = {TYPO_PROBABILITY}")
print(f"Q1 max word length        = {Q1_MAX_WORD_LEN}")
print(f"Q1 beam width             = {Q1_BEAM_SIZE}")

print("\nJustification:")
print("p = 0.08 gives realistic occasional fast-typing space omissions.")
print("N = 8 balances responsiveness with the cost of contextual grammar checks.")

Q4 configuration
-----------------
Merge probability p       = 0.08
Grammar trigger interval N = 8
Typo probability          = 0.1
Q1 max word length        = 18
Q1 beam width             = 25

Justification:
p = 0.08 gives realistic occasional fast-typing space omissions.
N = 8 balances responsiveness with the cost of contextual grammar checks.


In [36]:
# ============================================================
# PART 2 - PCFG TRAINING
# ============================================================

print("Loading Penn Treebank...")

treebank_trees = treebank.parsed_sents()

# Use the complete Penn Treebank for better lexical coverage.
PCFG_TRAIN_SIZE = len(treebank_trees)

pcfg_trees = []

for original_tree in treebank_trees[:PCFG_TRAIN_SIZE]:

    # Work on a copy so the original Treebank tree is unchanged.
    tree = original_tree.copy(deep=True)

    # Lowercase terminal words.
    for pos in tree.treepositions("leaves"):
        if isinstance(tree[pos], str):
            tree[pos] = tree[pos].lower()

    # Convert the tree into CNF form for CKY parsing.
    tree.chomsky_normal_form(horzMarkov=0)

    pcfg_trees.append(tree)

if not pcfg_trees:
    raise RuntimeError(
        "Penn Treebank is empty; cannot train the PCFG."
    )

# Collect all grammar productions.
productions = [
    prod
    for tree in pcfg_trees
    for prod in tree.productions()
]

# NLTK induce_pcfg() requires a Nonterminal object.
from nltk.grammar import Nonterminal

pcfg_start_symbol = Nonterminal(
    pcfg_trees[0].label()
)

# Induce the probabilistic context-free grammar.
pcfg_grammar = induce_pcfg(
    pcfg_start_symbol,
    productions
)

print(
    f"Treebank trees used: {PCFG_TRAIN_SIZE}"
)

print(
    f"Number of PCFG productions: "
    f"{len(pcfg_grammar.productions())}"
)

print(
    f"PCFG start symbol: "
    f"{pcfg_grammar.start().symbol()}"
)

print("PCFG training complete.")

Loading Penn Treebank...
Treebank trees used: 3914
Number of PCFG productions: 18998
PCFG start symbol: S
PCFG training complete.


In [37]:
# ============================================================
# PCFG INDEXES
# ============================================================

grammar_productions = pcfg_grammar.productions()

# Binary productions: A -> B C
binary_rules = defaultdict(list)

# Unary non-terminal productions: A -> B
unary_rules = defaultdict(list)

# Lexical productions: A -> 'word'
lexical_rules = defaultdict(list)

nonterminals = set()

for prod in grammar_productions:
    lhs = prod.lhs()
    rhs = prod.rhs()

    nonterminals.add(lhs)

    if len(rhs) == 2 and all(hasattr(x, "symbol") for x in rhs):
        binary_rules[(rhs[0], rhs[1])].append(
            (lhs, prod.prob())
        )

    elif len(rhs) == 1 and hasattr(rhs[0], "symbol"):
        unary_rules[rhs[0]].append(
            (lhs, prod.prob())
        )

    elif len(rhs) == 1:
        lexical_rules[str(rhs[0]).lower()].append(
            (lhs, prod.prob())
        )

print(f"Non-terminals: {len(nonterminals)}")
print(f"Binary rule groups: {len(binary_rules)}")
print(f"Unary rule groups: {len(unary_rules)}")
print(f"Known lexical items: {len(lexical_rules)}")

Non-terminals: 899
Binary rule groups: 3041
Unary rule groups: 45
Known lexical items: 11387


In [38]:
# ============================================================
# TAGSET RECONCILIATION
# ============================================================

Q1_TO_PTB = {
    "NOUN": "NN",
    "PROPN": "NNP",
    "VERB": "VB",
    "AUX": "VB",
    "ADJ": "JJ",
    "ADV": "RB",
    "PRON": "PRP",
    "DET": "DT",
    "ADP": "IN",
    "NUM": "CD",
    "CONJ": "CC",
    "CCONJ": "CC",
    "SCONJ": "IN",
    "PART": "RP",
    "INTJ": "UH",
    "SYM": "SYM",
    "X": "FW"
}


def reconcile_q1_tags(words):

    words = [
        w.lower()
        for w in words
    ]

    alpha_words = [
        w
        for w in words
        if w.isalpha()
    ]

    alpha_tags = hmm_en.tag(
        alpha_words
    )

    q1_tags = []
    ptb_tags = []

    j = 0

    for word in words:

        if word.isalpha():

            tag = alpha_tags[j]
            j += 1

            q1_tags.append(tag)
            ptb_tags.append(
                Q1_TO_PTB.get(tag)
            )

        else:

            q1_tags.append(None)
            ptb_tags.append(None)

    return q1_tags, ptb_tags

In [39]:
# ============================================================
# CKY / VITERBI PARSER
# ============================================================

def safe_log(prob):
    if prob <= 0:
        return -1e9
    return math.log(prob)


# Small penalty used when a word is not present in the
# Treebank lexical vocabulary but Q1 provides a plausible
# PTB preterminal tag.
UNKNOWN_WORD_PENALTY = -20.0


def apply_unary_closure(chart, i, j):
    """
    Add unary non-terminal expansions until no new
    higher-scoring constituent can be added.
    """
    changed = True

    while changed:
        changed = False

        current_items = list(
            chart[i][j].items()
        )

        for child, (score, tree) in current_items:

            for parent, prob in unary_rules.get(
                child,
                []
            ):

                new_score = (
                    score
                    + safe_log(prob)
                )

                if (
                    parent not in chart[i][j]
                    or new_score > chart[i][j][parent][0]
                ):

                    chart[i][j][parent] = (
                        new_score,
                        Tree(
                            parent.symbol(),
                            [tree]
                        )
                    )

                    changed = True


def cky_parse(
    words,
    q1_ptb_tags=None
):
    """
    Viterbi/CKY most-probable constituency parse.

    Q1 POS predictions are used as a soft lexical constraint.
    For words missing from the PCFG lexical vocabulary, a
    Q1-compatible PTB preterminal is used as an unknown-word
    lexical backoff with a penalty.

    A successful parse must span the complete sentence and
    must use the PCFG start symbol.
    """

    words = [
        str(w).lower()
        for w in words
        if str(w).strip()
    ]

    n = len(words)

    if n == 0:
        return None, -1e9

    chart = [
        [
            {}
            for _ in range(n + 1)
        ]
        for _ in range(n)
    ]

    # --------------------------------------------------------
    # Initialization: lexical productions
    # --------------------------------------------------------
    for i, word in enumerate(words):

        lexical_entries = list(
            lexical_rules.get(
                word,
                []
            )
        )

        preferred_tag = None

        if (
            q1_ptb_tags is not None
            and i < len(q1_ptb_tags)
        ):
            preferred_tag = q1_ptb_tags[i]

        # ----------------------------------------------------
        # Known word
        # ----        if lexical_entries:

            entries_to_use = list(lexical_entries)

            if preferred_tag is not None:

                from nltk.grammar import Nonterminal

                preferred_symbol = Nonterminal(preferred_tag)

                has_preferred = any(
                    item[0] == preferred_symbol
                    for item in entries_to_use
                )

                # Q1's tag isn't among this word's WSJ entries —
                # add it as a low-confidence extra option instead
                # of only accepting whatever tag WSJ happened to
                # give the word (e.g. "cat"/"sat" only ever show
                # up as NNP in the WSJ sample).
                if (
                    not has_preferred
                    and preferred_symbol in nonterminals
                ):
                    entries_to_use.append(
                        (preferred_symbol, None)
                    )

            for parent, prob in entries_to_use:

                score = (
                    UNKNOWN_WORD_PENALTY
                    if prob is None
                    else safe_log(prob)
                )

                leaf_tree = Tree(
                    parent.symbol(),
                    [word]
                )

                if (
                    parent not in chart[i][i + 1]
                    or
                    score
                    > chart[i][i + 1][parent][0]
                ):

                    chart[i][i + 1][parent] = (
                        score,
                        leaf_tree
                    )

        # ----------------------------------------------------
        # Unknown word
        # ----------------------------------------------------
        else:

            if preferred_tag is not None:

                from nltk.grammar import Nonterminal

                preferred_symbol = Nonterminal(
                    preferred_tag
                )

                # Use the Q1-compatible PTB tag only when it
                # is actually a known non-terminal in the grammar.
                if preferred_symbol in nonterminals:

                    chart[i][i + 1][
                        preferred_symbol
                    ] = (
                        UNKNOWN_WORD_PENALTY,
                        Tree(
                            preferred_tag,
                            [word]
                        )
                    )

        # If neither normal lexical entries nor a valid
        # unknown-word backoff exists, the sentence cannot
        # be parsed.
        if not chart[i][i + 1]:
            return None, -1e9

        apply_unary_closure(
            chart,
            i,
            i + 1
        )

    # --------------------------------------------------------
    # CKY dynamic programming
    # --------------------------------------------------------
    for span in range(
        2,
        n + 1
    ):

        for i in range(
            0,
            n - span + 1
        ):

            j = i + span

            for k in range(
                i + 1,
                j
            ):

                left_items = list(
                    chart[i][k].items()
                )

                right_items = list(
                    chart[k][j].items()
                )

                for (
                    left_symbol,
                    (
                        left_score,
                        left_tree
                    )
                ) in left_items:

                    for (
                        right_symbol,
                        (
                            right_score,
                            right_tree
                        )
                    ) in right_items:

                        candidates = binary_rules.get(
                            (
                                left_symbol,
                                right_symbol
                            ),
                            []
                        )

                        for parent, prob in candidates:

                            score = (
                                left_score
                                + right_score
                                + safe_log(prob)
                            )

                            if (
                                parent not in chart[i][j]
                                or
                                score
                                > chart[i][j][parent][0]
                            ):

                                chart[i][j][parent] = (
                                    score,
                                    Tree(
                                        parent.symbol(),
                                        [
                                            left_tree,
                                            right_tree
                                        ]
                                    )
                                )

            apply_unary_closure(
                chart,
                i,
                j
            )

    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------
    root = pcfg_grammar.start()

    if root in chart[0][n]:

        return (
            chart[0][n][root][1],
            chart[0][n][root][0]
        )

    # Do not accept a partial constituent as a sentence parse.
    return None, -1e9

In [40]:
# ============================================================
# TEST CKY PARSER
# ============================================================

test_sentences = [
    "the cat sat on the mat",
    "she eats a green salad",
    "i saw the man with a telescope"
]

for sentence in test_sentences:

    words = sentence.split()

    q1_tags, ptb_tags = reconcile_q1_tags(words)

    tree, score = cky_parse(
        words,
        q1_ptb_tags=ptb_tags
    )

    print("\nSentence:", sentence)

    if tree is None:
        print("UNPARSEABLE")
    else:
        print("Parse score:", score)
        print(tree)


Sentence: the cat sat on the mat
Parse score: -56.22354923085897
(S
  (NP-SBJ (DT the) (NP-SBJ|<> (NNP cat) (NNP sat)))
  (PP-PRD (IN on) (NP (DT the) (NN mat))))

Sentence: she eats a green salad
Parse score: -61.35170296460404
(S
  (NP-SBJ (PRP she))
  (VP (VB eats) (NP (DT a) (NP|<> (JJ green) (NN salad)))))

Sentence: i saw the man with a telescope
Parse score: -54.81465900714462
(S
  (NP-SBJ (PRP i))
  (VP
    (VBD saw)
    (VP|<>
      (NP (DT the) (NN man))
      (PP-CLR (IN with) (NP (DT a) (NN telescope))))))


In [41]:
# ============================================================
# PCFG LEXICAL COVERAGE CHECK
# ============================================================

check_words = [
    "the",
    "cat",
    "sat",
    "on",
    "mat",
    "she",
    "eats",
    "a",
    "green",
    "salad",
    "i",
    "saw",
    "man",
    "with",
    "telescope"
]

print("PCFG start symbol:", pcfg_grammar.start())
print("Number of grammar productions:", len(pcfg_grammar.productions()))
print("Number of lexical items:", len(lexical_rules))
print()

for word in check_words:

    entries = lexical_rules.get(word, [])

    if entries:
        tags = sorted(
            set(parent.symbol() for parent, prob in entries)
        )
        print(f"{word:12s} -> FOUND    {tags}")
    else:
        print(f"{word:12s} -> MISSING")

PCFG start symbol: S
Number of grammar productions: 18998
Number of lexical items: 11387

the          -> FOUND    ['CD', 'DT', 'JJ', 'NNP']
cat          -> FOUND    ['NNP']
sat          -> FOUND    ['NNP']
on           -> FOUND    ['IN', 'NNP', 'RB', 'RP']
mat          -> MISSING
she          -> FOUND    ['PRP']
eats         -> MISSING
a            -> FOUND    ['DT', 'IN', 'JJ', 'LS', 'NN', 'NNP']
green        -> FOUND    ['JJ', 'NNP']
salad        -> MISSING
i            -> FOUND    ['NNP', 'PRP']
saw          -> FOUND    ['VBD']
man          -> FOUND    ['NN', 'NNP']
with         -> FOUND    ['IN']
telescope    -> MISSING


In [42]:
# ============================================================
# SENTENCE-LEVEL BIGRAM / TRIGRAM SCORING
# ============================================================

# Reuse Q3's trained bigram model
Q4_K = 0.5

q4_bigram_freq = bigram_freq
q4_unigram_ctx_freq = unigram_ctx_freq
q4_vocab_size = len(vocab) + 2


def q4_bigram_prob(w1, w2, k=Q4_K):

    return (
        q4_bigram_freq.get(
            (w1, w2),
            0
        ) + k
    ) / (
        q4_unigram_ctx_freq.get(
            w1,
            0
        ) + k * q4_vocab_size
    )


def lm_words(sentence):

    tokens = nltk.word_tokenize(
        sentence.lower()
    )

    return [
        word
        for word in tokens
        if word.isalpha()
    ]


def sentence_bigram_score(sentence):

    words = lm_words(sentence)

    tokens = (
        ["<s>"]
        + words
        + ["</s>"]
    )

    score = 0.0

    for i in range(
        len(tokens) - 1
    ):

        probability = q4_bigram_prob(
            tokens[i],
            tokens[i + 1]
        )

        score += math.log(
            max(
                probability,
                1e-12
            )
        )

    return score


def sentence_trigram_score(sentence):

    words = lm_words(sentence)

    if not words:
        return 0.0

    score = 0.0

    previous_previous = "<s>"
    previous = "<s>"

    # --------------------------------------------------------
    # Reuse Q1's trained English trigram model
    # --------------------------------------------------------

    for word in words:

        probability = lm_en.word_probability(
            previous_previous,
            previous,
            word
        )

        score += math.log(
            max(
                probability,
                1e-12
            )
        )

        previous_previous = previous
        previous = word

    # End-of-sentence probability from Q1 LM
    end_probability = lm_en.word_probability(
        previous_previous,
        previous,
        "</s>"
    )

    score += math.log(
        max(
            end_probability,
            1e-12
        )
    )

    return score


def sentence_perplexity(
    log_probability,
    token_count
):

    if token_count <= 0:
        return float("inf")

    return math.exp(
        -log_probability / token_count
    )


# ------------------------------------------------------------
# Test the shared language models
# ------------------------------------------------------------

example = "the cat sat on the mat"

b_score = sentence_bigram_score(
    example
)

t_score = sentence_trigram_score(
    example
)

print(
    "Sentence:",
    example
)

print(
    "Bigram log probability:",
    b_score
)

print(
    "Trigram log probability:",
    t_score
)

print(
    "Bigram perplexity:",
    sentence_perplexity(
        b_score,
        len(lm_words(example)) + 1
    )
)

print(
    "Trigram perplexity:",
    sentence_perplexity(
        t_score,
        len(lm_words(example)) + 1
    )
)

Sentence: the cat sat on the mat
Bigram log probability: -53.66303968383465
Trigram log probability: -43.800093023878645
Bigram perplexity: 2134.843298030635
Trigram perplexity: 521.7330957361476


In [43]:
# ============================================================
# PART 1 - SIMULATED FAST-TYPING  (missed spaces + typos)
# ============================================================

LETTERS = "abcdefghijklmnopqrstuvwxyz"


def generate_merged_tokens(tokens, p=MERGE_PROBABILITY):
    """Kept for compatibility: drop the space between adjacent words."""
    if not tokens:
        return []
    result, i = [], 0
    while i < len(tokens):
        if (i < len(tokens) - 1 and tokens[i].isalpha()
                and tokens[i + 1].isalpha() and random.random() < p):
            result.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return result


def corrupt_word(word):
    """Return a NON-WORD typo of `word` (delete/substitute/transpose/insert)."""
    for _ in range(20):
        i = random.randrange(len(word))
        op = random.choice(["delete", "substitute", "transpose", "insert"])
        if op == "delete":
            t = word[:i] + word[i + 1:]
        elif op == "substitute":
            t = word[:i] + random.choice(LETTERS) + word[i + 1:]
        elif op == "transpose" and i < len(word) - 1:
            t = word[:i] + word[i + 1] + word[i] + word[i + 2:]
        else:
            t = word[:i] + random.choice(LETTERS) + word[i:]
        if t != word and t not in vocab and t not in vocab_en:
            return t
    return None


def simulate_typed_stream(tokens, p_merge=MERGE_PROBABILITY,
                          p_typo=TYPO_PROBABILITY):
    """
    Turn clean tokens into what a fast, sloppy typist produces.
    Returns [{"typed", "expected", "kind"}] where kind is
    "merge" (space dropped), "typo" (word mistyped) or "clean".
    `expected` is the ground truth the pipeline should recover.
    """
    out, i = [], 0
    while i < len(tokens):
        tok = tokens[i]
        low = tok.lower()

        if (i < len(tokens) - 1 and tok.isalpha() and tokens[i + 1].isalpha()
                and random.random() < p_merge):
            nxt = tokens[i + 1].lower()
            # skip if the merge accidentally forms a valid word
            if (low + nxt) not in vocab and (low + nxt) not in vocab_en:
                out.append({"typed": low + nxt,
                            "expected": [low, nxt], "kind": "merge"})
                i += 2
                continue

        if (tok.isalpha() and len(tok) >= 4 and low in vocab
                and random.random() < p_typo):
            t = corrupt_word(low)
            if t:
                out.append({"typed": t, "expected": [low], "kind": "typo"})
                i += 1
                continue

        out.append({"typed": low, "expected": [low], "kind": "clean"})
        i += 1
    return out


In [44]:
# ============================================================
# SPELLING ALERT
# ============================================================

def spelling_check(token):

    token = token.lower()

    # Punctuation is not a spelling error
    if not token.isalpha():

        return {
            "changed": False,
            "word": token
        }

    if token in vocab:

        return {
            "changed": False,
            "word": token
        }

    # Reuse Q3's trained corrector
    corrected = correct_nonword(
        token,
        method="B"
    )

    return {
        "changed": corrected != token,
        "word": corrected
    }

In [45]:
# ============================================================
# TEST SPELLING ALERT
# ============================================================

for token in [
    "hav",
    "sentnce",
    "good"
]:

    print(
        token,
        "=>",
        spelling_check(token)
    )

hav => {'changed': True, 'word': 'had'}
sentnce => {'changed': True, 'word': 'sentence'}
good => {'changed': False, 'word': 'good'}


In [46]:
# ============================================================
# GRAMMAR + REAL-WORD ALERT
# ============================================================

# A real-word correction must improve the local bigram score
# by at least this amount before it is reported.
REAL_WORD_MIN_LOG_GAIN = 2.0

# Grammar-alert thresholds.
BIGRAM_ALERT_THRESHOLD = -10.0
TRIGRAM_ALERT_THRESHOLD = -12.0


def local_bigram_log_score(
    words,
    index
):
    """
    Calculate the local bigram log-probability involving
    the word at the given index.
    """

    if index < 0 or index >= len(words):
        return -1e9

    word = words[index]

    score = 0.0

    # Left context
    if index == 0:
        score += math.log(
            max(
                q4_bigram_prob(
                    "<s>",
                    word
                ),
                1e-12
            )
        )
    else:
        score += math.log(
            max(
                q4_bigram_prob(
                    words[index - 1],
                    word
                ),
                1e-12
            )
        )

    # Right context
    if index == len(words) - 1:
        score += math.log(
            max(
                q4_bigram_prob(
                    word,
                    "</s>"
                ),
                1e-12
            )
        )
    else:
        score += math.log(
            max(
                q4_bigram_prob(
                    word,
                    words[index + 1]
                ),
                1e-12
            )
        )

    return score


def grammar_trigger_check(words):

    window = [
        str(word).lower()
        for word in words
        if str(word).strip()
    ][-GRAMMAR_TRIGGER_N:]

    if len(window) < 2:
        return {
            "grammar_alert": False,
            "real_word_alerts": [],
            "bigram_score": 0.0,
            "trigram_score": 0.0,
            "avg_bigram": 0.0,
            "avg_trigram": 0.0
        }

    sentence = " ".join(window)

    # Shared Q4 language-model scores
    b_score = sentence_bigram_score(sentence)
    t_score = sentence_trigram_score(sentence)

    # Normalize by the number of scored transitions.
    avg_bigram = (
        b_score / max(len(window) + 1, 1)
    )

    avg_trigram = (
        t_score / max(len(window) + 1, 1)
    )

    real_word_alerts = []

    # --------------------------------------------------------
    # Real-word error detection
    # --------------------------------------------------------

    for i, word in enumerate(window):

        if not word.isalpha():
            continue

        if word not in vocab:
            continue

        corrected, changed = correct_realword(
            window,
            i,
            method="B"
        )

        if not changed:
            continue

        if not corrected:
            continue

        corrected = str(corrected).lower()

        if corrected == word:
            continue

        # Score original word in its context.
        original_local_score = local_bigram_log_score(
            window,
            i
        )

        # Score Q3's proposed correction in the same context.
        candidate_window = list(window)
        candidate_window[i] = corrected

        candidate_local_score = local_bigram_log_score(
            candidate_window,
            i
        )

        log_gain = (
            candidate_local_score
            - original_local_score
        )

        # Only flag the correction when the contextual
        # improvement is substantial.
        if log_gain >= REAL_WORD_MIN_LOG_GAIN:
            real_word_alerts.append({
                "index": i,
                "original": word,
                "suggestion": corrected,
                "log_gain": log_gain
            })

    # --------------------------------------------------------
    # Grammar alert
    # --------------------------------------------------------

    grammar_alert = (
        avg_bigram < BIGRAM_ALERT_THRESHOLD
        or
        avg_trigram < TRIGRAM_ALERT_THRESHOLD
    )

    return {
        "grammar_alert": grammar_alert,
        "real_word_alerts": real_word_alerts,
        "bigram_score": b_score,
        "trigram_score": t_score,
        "avg_bigram": avg_bigram,
        "avg_trigram": avg_trigram
    }

In [47]:
# ============================================================
# SEGMENTATION ALERT
# ============================================================

def segmentation_check(token):

    token = token.lower()

    # Punctuation is not a segmentation candidate
    if not token.isalpha():
        return {
            "changed": False,
            "words": [token],
            "tags": [None]
        }

    # Known normal-length word
    if (
        token in vocab_en
        and len(token) <= Q1_MAX_WORD_LEN
    ):
        return {
            "changed": False,
            "words": [token],
            "tags": hmm_en.tag([token])
        }

    # Reuse Q1 trained English segmentation model
    split_words = lm_en.segment(token)

    # Only report a segmentation alert when
    # a genuine multi-word split is found.
    if (
        len(split_words) > 1
        and all(
            word in vocab_en
            for word in split_words
        )
    ):

        tags = hmm_en.tag(
            split_words
        )

        return {
            "changed": True,
            "words": split_words,
            "tags": tags
        }

    return {
        "changed": False,
        "words": [token],
        "tags": hmm_en.tag([token])
    }

VALID_TWO = {
    "of", "to", "in", "it", "is", "be", "he", "we", "me", "my", "no",
    "so", "do", "go", "up", "us", "on", "or", "an", "as", "at", "by",
    "if", "am", "oh", "ok", "hi", "ma", "pa", "ah", "ha", "eh", "ex",
    "id", "la", "lo", "mr", "ms", "pi", "re", "yo", "ye",
}


def good_piece(word):
    if len(word) == 1:
        return word in {"a", "i"}
    if len(word) == 2:
        return word in VALID_TWO
    return word in vocab_en and word in vocab


def plausible_split(parts):
    return len(parts) > 1 and all(good_piece(p) for p in parts)


def resolve_token(token):
    token = str(token).lower()

    if not token or not token.isalpha():
        return {"kind": "keep", "words": [token] if token else []}

    if token in vocab or token in vocab_en:
        return {"kind": "keep", "words": [token]}

    parts = lm_en.segment(token)
    if plausible_split(parts):
        return {"kind": "segment", "words": parts}

    corrected = correct_nonword(token, method="B")
    if corrected != token:
        return {"kind": "spell", "words": [corrected]}

    return {"kind": "keep", "words": [token]}


In [48]:
# ============================================================
# INTEGRATED LIVE PIPELINE
# ============================================================

def process_live_token(
    token,
    accumulated_words
):

    events = []

    start = time.perf_counter()

    # --------------------------------------------------------
    # Normalize incoming token
    # --------------------------------------------------------

    token = str(token).lower()

    words_before = len(
        accumulated_words
    )

    # --------------------------------------------------------
    # STEP 1+2: SEGMENTATION or SPELLING (one decision)
    # --------------------------------------------------------

    resolved = resolve_token(token)
    final_words = resolved["words"]

    if resolved["kind"] == "segment":

        events.append({
            "type": "SEGMENT-ALERT",
            "message": (
                f"{token} -> "
                + " | ".join(final_words)
                + " | POS="
                + str(hmm_en.tag(final_words))
            )
        })

    elif resolved["kind"] == "spell":

        events.append({
            "type": "SPELL-ALERT",
            "message": f"{token} -> {final_words[0]}"
        })

    # --------------------------------------------------------
    # Add processed words
    # --------------------------------------------------------

    accumulated_words.extend(
        final_words
    )

    words_after = len(
        accumulated_words
    )

    per_token_latency = (
        time.perf_counter() - start
    ) * 1000

    # --------------------------------------------------------
    # STEP 3: GRAMMAR + REAL-WORD TRIGGER
    # --------------------------------------------------------

    grammar_result = None
    grammar_latency = 0.0

    next_trigger = (
        (
            words_before
            // GRAMMAR_TRIGGER_N
        ) + 1
    ) * GRAMMAR_TRIGGER_N

    crossed_trigger = (
        words_after >= next_trigger
    )

    if crossed_trigger:

        grammar_start = time.perf_counter()

        window = accumulated_words[
            -GRAMMAR_TRIGGER_N:
        ]

        grammar_result = (
            grammar_trigger_check(
                window
            )
        )

        grammar_latency = (
            time.perf_counter()
            - grammar_start
        ) * 1000

        if grammar_result[
            "grammar_alert"
        ]:

            events.append({
                "type": "GRAMMAR-ALERT",
                "message": (
                    "Implausible recent word "
                    "sequence "
                    f"(bigram avg="
                    f"{grammar_result['avg_bigram']:.2f}, "
                    f"trigram avg="
                    f"{grammar_result['avg_trigram']:.2f})"
                )
            })

        for alert in grammar_result[
            "real_word_alerts"
        ]:

            events.append({
                "type": "GRAMMAR-ALERT",
                "message": (
                    "Possible real-word error: "
                    f"{alert['original']} -> "
                    f"{alert['suggestion']} "
                    f"(log gain="
                    f"{alert['log_gain']:.2f})"
                )
            })

    return {
        "words": final_words,
        "kind": resolved["kind"],
        "events": events,
        "token_latency_ms": per_token_latency,
        "grammar_latency_ms": grammar_latency,
        "grammar_result": grammar_result
    }

In [49]:
# ============================================================
# RANDOM CONTIGUOUS PASSAGE GENERATOR
# ============================================================

def sample_random_passage(
    min_sentences=5,
    max_sentences=8,
    min_words=5,
    max_words=25
):

    files = gutenberg.fileids()

    random.shuffle(files)

    for file_id in files:

        raw_text = gutenberg.raw(
            file_id
        )

        sentences = nltk.sent_tokenize(
            raw_text
        )

        sentences = [
            s.strip()
            for s in sentences
        ]

        possible_windows = []

        for n in range(
            min_sentences,
            max_sentences + 1
        ):

            for start in range(
                0,
                len(sentences) - n + 1
            ):

                window = sentences[
                    start:start + n
                ]

                valid = all(
                    min_words
                    <= len(s.split())
                    <= max_words
                    for s in window
                )

                if valid:
                    possible_windows.append(
                        window
                    )

        if possible_windows:

            selected = random.choice(
                possible_windows
            )

            return (
                file_id,
                selected
            )

    raise RuntimeError(
        "Could not find a contiguous Gutenberg passage "
        "with 5–8 sentences within the requested length."
    )


file_id, passage_sentences = (
    sample_random_passage()
)

print(
    "Source:",
    file_id
)

print(
    "\nPASSAGE\n-------"
)

for i, s in enumerate(
    passage_sentences,
    start=1
):

    print(
        f"{i}. {s}"
    )

print(
    "\nNumber of sentences:",
    len(passage_sentences)
)

print(
    "Sentence word counts:",
    [
        len(s.split())
        for s in passage_sentences
    ]
)

assert 5 <= len(passage_sentences) <= 8

Source: whitman-leaves.txt

PASSAGE
-------
1. I am he that walks with the tender and growing night,
I call to the earth and sea half-held by the night.
2. Press close bare-bosom'd night--press close magnetic nourishing night!
3. Night of south winds--night of the large few stars!
4. Still nodding night--mad naked summer night.
5. Smile O voluptuous cool-breath'd earth!
6. Earth of the slumbering and liquid trees!
7. Earth of departed sunset--earth of the mountains misty-topt!
8. Earth of the vitreous pour of the full moon just tinged with blue!

Number of sentences: 8
Sentence word counts: [22, 8, 9, 6, 5, 7, 8, 13]


In [50]:
# ============================================================
# LIVE TYPING SIMULATION  (missed spaces + typos)
# ============================================================

def simulate_live_typing(
    sentences,
    delay=0.03,
    p_merge=MERGE_PROBABILITY,
    p_typo=TYPO_PROBABILITY
):

    processed_words = []

    token_latencies = []
    grammar_latencies = []

    segmentation_merges = 0
    spelling_corrections = 0

    injected = {"merge": 0, "typo": 0}
    recovered = {"merge": 0, "typo": 0}

    sentence_records = []

    print("\n" + "=" * 70)
    print("LIVE TYPING SIMULATION")
    print("=" * 70)

    for sentence in sentences:

        stream = simulate_typed_stream(
            nltk.word_tokenize(sentence),
            p_merge=p_merge,
            p_typo=p_typo
        )

        sentence_start_index = len(processed_words)
        sentence_merges = 0
        sentence_spells = 0

        for item in stream:

            token = item["typed"]

            result = process_live_token(
                token,
                processed_words
            )

            token_latencies.append(result["token_latency_ms"])

            if result["grammar_latency_ms"] > 0:
                grammar_latencies.append(result["grammar_latency_ms"])

            for event in result["events"]:

                print(f"[{event['type']}] {event['message']}")

                if event["type"] == "SEGMENT-ALERT":
                    segmentation_merges += 1
                    sentence_merges += 1

                elif event["type"] == "SPELL-ALERT":
                    spelling_corrections += 1
                    sentence_spells += 1

            # ground-truth check for injected errors
            if item["kind"] in injected:
                injected[item["kind"]] += 1
                if result["words"] == item["expected"]:
                    recovered[item["kind"]] += 1

            tag = "" if item["kind"] == "clean" else f"  (injected {item['kind']})"
            print("Typed:", token, " | Processed:",
                  " ".join(result["words"]) + tag)

            time.sleep(delay)

        sentence_records.append({
            "original_sentence": sentence,
            "processed_words": processed_words[sentence_start_index:],
            "segmentation_merges": sentence_merges,
            "spelling_corrections": sentence_spells
        })

    print("\n" + "-" * 70)
    print("LIVE TYPING SUMMARY")
    print("-" * 70)

    print("Processed words        :", len(processed_words))
    print("Segmentation merges    :", segmentation_merges)
    print("Spelling corrections   :", spelling_corrections)
    print(f"Injected missed spaces : {injected['merge']}  "
          f"(recovered exactly: {recovered['merge']})")
    print(f"Injected typos         : {injected['typo']}  "
          f"(corrected exactly: {recovered['typo']})")
    print("Token checks           :", len(token_latencies))
    print("Grammar triggers       :", len(grammar_latencies))

    if token_latencies:
        print("Average token latency  :",
              f"{statistics.mean(token_latencies):.4f} ms")

    if grammar_latencies:
        print("Average grammar latency:",
              f"{statistics.mean(grammar_latencies):.4f} ms")

    return {
        "words": processed_words,
        "token_latencies": token_latencies,
        "grammar_latencies": grammar_latencies,
        "segmentation_merges": segmentation_merges,
        "spelling_corrections": spelling_corrections,
        "injected": injected,
        "recovered": recovered,
        "sentence_records": sentence_records
    }


live_result = simulate_live_typing(
    passage_sentences
)



LIVE TYPING SIMULATION
Typed: i  | Processed: i
Typed: am  | Processed: am
Typed: he  | Processed: he
Typed: that  | Processed: that
Typed: walks  | Processed: walks
Typed: with  | Processed: with
Typed: the  | Processed: the
[GRAMMAR-ALERT] Possible real-word error: walks -> talks (log gain=2.40)
Typed: tender  | Processed: tender
Typed: and  | Processed: and
[SEGMENT-ALERT] growingnight -> growing | night | POS=['VERB', 'NOUN']
Typed: growingnight  | Processed: growing night  (injected merge)
Typed: ,  | Processed: ,
[SEGMENT-ALERT] icall -> i | call | POS=['PRON', 'VERB']
Typed: icall  | Processed: i call  (injected merge)
Typed: to  | Processed: to
Typed: the  | Processed: the
Typed: earth  | Processed: earth
Typed: and  | Processed: and
Typed: sea  | Processed: sea
Typed: half-held  | Processed: half-held
Typed: by  | Processed: by
Typed: the  | Processed: the
Typed: night  | Processed: night
[GRAMMAR-ALERT] Possible real-word error: sea -> she (log gain=3.50)
Typed: .  | Process

In [51]:
# ============================================================
# LATENCY REPORT
# ============================================================

token_latencies = live_result[
    "token_latencies"
]

grammar_latencies = [
    x
    for x in live_result["grammar_latencies"]
    if x > 0
]

avg_token_latency = (
    statistics.mean(token_latencies)
    if token_latencies
    else 0
)

avg_grammar_latency = (
    statistics.mean(grammar_latencies)
    if grammar_latencies
    else 0
)

print("=" * 60)
print("LATENCY REPORT")
print("=" * 60)

print(
    f"Average segmentation + spelling latency: "
    f"{avg_token_latency:.3f} ms"
)

print(
    f"Average grammar trigger latency: "
    f"{avg_grammar_latency:.3f} ms"
)

print(
    f"Segmentation merges resolved: "
    f"{live_result['segmentation_merges']}"
)

LATENCY REPORT
Average segmentation + spelling latency: 0.047 ms
Average grammar trigger latency: 0.676 ms
Segmentation merges resolved: 5


In [52]:
# ============================================================
# PART 4 - FINAL PASSAGE ANALYSIS
# ============================================================

def analyze_sentence(sentence):
    """
    Analyze one final corrected sentence using:
      1. Q1 English POS + tag reconciliation
      2. Q4 PCFG / CKY parser
      3. Q3 bigram model
      4. Q1 trigram model
    """

    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    raw_words = nltk.word_tokenize(
        sentence.lower()
    )

    parse_words = [
        word
        for word in raw_words
        if word.isalpha()
    ]

    if not parse_words:
        return {
            "sentence": sentence,
            "pcfg_result": "UNPARSEABLE",
            "pcfg_score": -1e9,
            "bigram_score": -1e9,
            "trigram_score": -1e9,
            "chosen_method": "Bigram",
            "final_verdict": "Possibly ungrammatical"
        }

    # --------------------------------------------------------
    # Q1 POS -> Penn Treebank tag reconciliation
    # --------------------------------------------------------

    q1_tags, ptb_tags = reconcile_q1_tags(
        parse_words
    )

    # --------------------------------------------------------
    # PCFG / CKY
    # --------------------------------------------------------

    tree, pcfg_score = cky_parse(
        parse_words,
        q1_ptb_tags=ptb_tags
    )

    if tree is None:
        pcfg_result = "UNPARSEABLE"
    else:
        pcfg_result = "PARSED"

    # --------------------------------------------------------
    # N-gram scores
    # --------------------------------------------------------

    bigram_score = sentence_bigram_score(
        " ".join(parse_words)
    )

    trigram_score = sentence_trigram_score(
    " ".join(parse_words)
    )

    token_count = max(
        len(parse_words) + 1,
        1
    )

    trigram_avg = (
        trigram_score / token_count
    )

    if pcfg_result == "PARSED":

        chosen_method = "PCFG"
        verdict = "Likely grammatical"

    elif trigram_avg >= TRIGRAM_ALERT_THRESHOLD:

        chosen_method = "Trigram"
        verdict = "Likely grammatical"

    else:

        chosen_method = "Bigram"
        verdict = "Possibly ungrammatical"

    return {
        "sentence": sentence,
        "pcfg_result": pcfg_result,
        "pcfg_score": pcfg_score,
        "bigram_score": bigram_score,
        "trigram_score": trigram_score,
        "chosen_method": chosen_method,
        "final_verdict": verdict
    }

In [53]:
# ============================================================
# FINAL PASSAGE ANALYSIS WITH CORRECTION COUNTS
# ============================================================

def analyze_passage_with_counters(
    sentences
):

    rows = []

    for sentence in sentences:
        stream = simulate_typed_stream(
            nltk.word_tokenize(sentence),
            p_merge=MERGE_PROBABILITY,
            p_typo=TYPO_PROBABILITY
        )

        processed_words = []

        merge_count = 0
        spelling_count = 0

        # ----------------------------------------------------
        # Segmentation OR spelling (one decision per token)
        # ----------------------------------------------------

        for item in stream:

            resolved = resolve_token(item["typed"])

            if resolved["kind"] == "segment":
                merge_count += 1
            elif resolved["kind"] == "spell":
                spelling_count += 1

            processed_words.extend(resolved["words"])

        # ----------------------------------------------------
        # Final corrected sentence
        # ----------------------------------------------------

        final_sentence = " ".join(
            processed_words
        )

        result = analyze_sentence(
            final_sentence
        )

        result["segmentation_merges"] = (
            merge_count
        )

        result["spelling_corrections"] = (
            spelling_count
        )

        rows.append(result)

    return pd.DataFrame(rows)


# Analyze the complete 5–8 sentence passage
final_analysis_df = (
    analyze_passage_with_counters(
        passage_sentences
    )
)

display(final_analysis_df)

print(
    "\nNumber of analyzed sentences:",
    len(final_analysis_df)
)

assert 5 <= len(final_analysis_df) <= 8

,sentence,pcfg_result,pcfg_score,bigram_score,trigram_score,chosen_method,final_verdict,segmentation_merges,spelling_corrections
0,i am he that walks with the tender and growing...,PARSED,-168.502317,-167.189324,-131.915009,PCFG,Likely grammatical,1,0
1,press close bare-bosom 'd night -- press close...,PARSED,-92.072444,-88.993107,-99.955229,PCFG,Likely grammatical,0,0
2,might of south winds -- night of the large few...,PARSED,-92.348866,-94.652310,-79.957640,PCFG,Likely grammatical,2,1
3,still nod dig night -- mad naked summer night .,PARSED,-135.452773,-86.986218,-87.790910,PCFG,Likely grammatical,1,1
4,smile voluptuous cool-breath 'd earth !,PARSED,-75.487306,-39.401381,-37.734333,PCFG,Likely grammatical,0,1
5,earth of the lumbering and liquid trees !,PARSED,-87.993751,-71.024416,-53.471196,PCFG,Likely grammatical,0,1
6,earth of departed sunset -- earth of the mount...,PARSED,-129.606554,-83.190788,-67.986087,PCFG,Likely grammatical,0,0
7,earth of the vitreous pour of the full moon ju...,PARSED,-157.794489,-124.456022,-119.068266,PCFG,Likely grammatical,0,1



Number of analyzed sentences: 8


In [54]:
# ============================================================
# INTERACTION ANALYSIS
# ============================================================

print("=" * 70)
print("Q4 INTERACTION ANALYSIS")
print("=" * 70)

total_sentences = len(
    final_analysis_df
)

seg_corrections = (
    final_analysis_df[
        "segmentation_merges"
    ].sum()
)

spell_corrections = (
    final_analysis_df[
        "spelling_corrections"
    ].sum()
)

pcfg_parsed = (
    final_analysis_df[
        "pcfg_result"
    ] == "PARSED"
).sum()

print(
    f"Total sentences analyzed       : "
    f"{total_sentences}"
)

print(
    f"Segmentation merges corrected  : "
    f"{seg_corrections}"
)

print(
    f"Spelling corrections applied   : "
    f"{spell_corrections}"
)

print(
    f"Sentences successfully parsed  : "
    f"{pcfg_parsed}/{total_sentences}"
)

print(
    "\nInterpretation:"
)

print(
    "Segmentation and spelling corrections are performed "
    "before the PCFG analysis, so an upstream correction "
    "can change whether a sentence is parseable and can "
    "also change the n-gram method selected at the end."
)

Q4 INTERACTION ANALYSIS
Total sentences analyzed       : 8
Segmentation merges corrected  : 4
Spelling corrections applied   : 5
Sentences successfully parsed  : 8/8

Interpretation:
Segmentation and spelling corrections are performed before the PCFG analysis, so an upstream correction can change whether a sentence is parseable and can also change the n-gram method selected at the end.


In [55]:
# ============================================================
# PART 5 - SPEED DEMON BENCHMARK
# ============================================================

# ------------------------------------------------------------
# Create a simple synthetic non-word corruption
# ------------------------------------------------------------

def introduce_error(word):

    word = str(word).lower()

    if len(word) < 2:
        return word + "x"

    # Delete one character from the middle.
    # This creates a non-word while remaining realistic
    # for spelling-correction benchmarking.
    pos = random.randint(
        0,
        len(word) - 1
    )

    corrupted = (
        word[:pos]
        + word[pos + 1:]
    )

    # Make sure the result is actually different.
    if corrupted == word:
        corrupted = word + "x"

    return corrupted


# ------------------------------------------------------------
# Build exactly 1,000 corrupted non-word tokens
# ------------------------------------------------------------

vocab_list_q4 = [
    word
    for word in vocab
    if str(word).isalpha()
]

benchmark_words = []

while len(benchmark_words) < 1000:

    original = random.choice(
        vocab_list_q4
    )

    corrupted = introduce_error(
        original
    )

    if corrupted not in vocab:

        benchmark_words.append(
            corrupted
        )

assert len(benchmark_words) == 1000

print(
    "Verified benchmark size:",
    len(benchmark_words)
)

Verified benchmark size: 1000


In [56]:
# ============================================================
# REQUIRED TEST SENTENCES
# ============================================================

required_tests = [
    "The cat sat on the mat.",
    "She eats a green salad.",
    "I saw the man with a telescope.",
    "I hav a good feeling about this.",
    "This is a test sentnce.",
    "I would like to sea the world.",
    "Please meat me at the station."
]

for sentence in required_tests:

    print("\n" + "=" * 70)
    print("INPUT:", sentence)

    words = sentence.lower().split()

    result = analyze_sentence(
        " ".join(words)
    )

    print(
        "PCFG:",
        result["pcfg_result"]
    )

    print(
        "Bigram:",
        result["bigram_score"]
    )

    print(
        "Trigram:",
        result["trigram_score"]
    )

    print(
        "Chosen:",
        result["chosen_method"]
    )

    print(
        "Verdict:",
        result["final_verdict"]
    )


INPUT: The cat sat on the mat.
PCFG: PARSED
Bigram: -53.66303968383465
Trigram: -43.800093023878645
Chosen: PCFG
Verdict: Likely grammatical

INPUT: She eats a green salad.
PCFG: PARSED
Bigram: -55.909853672041145
Trigram: -50.30500967659583
Chosen: PCFG
Verdict: Likely grammatical

INPUT: I saw the man with a telescope.
PCFG: PARSED
Bigram: -51.606500036186816
Trigram: -37.032685322618285
Chosen: PCFG
Verdict: Likely grammatical

INPUT: I hav a good feeling about this.
PCFG: PARSED
Bigram: -59.04678594063677
Trigram: -60.09816095906149
Chosen: PCFG
Verdict: Likely grammatical

INPUT: This is a test sentnce.
PCFG: PARSED
Bigram: -41.76580256788056
Trigram: -50.07720754454767
Chosen: PCFG
Verdict: Likely grammatical

INPUT: I would like to sea the world.
PCFG: PARSED
Bigram: -50.212493298084546
Trigram: -34.01113223915203
Chosen: PCFG
Verdict: Likely grammatical

INPUT: Please meat me at the station.
PCFG: PARSED
Bigram: -55.79069410661702
Trigram: -45.69381067678786
Chosen: PCFG
Verdi

In [57]:
# ============================================================
# SAVE Q4 MODEL BUNDLE
# ============================================================

import cloudpickle

q4_bundle = {

    # --------------------------------------------------------
    # Q1
    # --------------------------------------------------------

    "lm_en": lm_en,
    "hmm_en": hmm_en,
    "vocab_en": vocab_en,

    # --------------------------------------------------------
    # Q3
    # --------------------------------------------------------

    "vocab": vocab,
    "unigram_freq": unigram_freq,
    "bigram_freq": bigram_freq,
    "unigram_ctx_freq": unigram_ctx_freq,
    "symspell_index": symspell_index,

    "correct_nonword": correct_nonword,
    "correct_realword": correct_realword,

    # --------------------------------------------------------
    # Q4 PCFG
    # --------------------------------------------------------

    "pcfg_grammar": pcfg_grammar,
    "binary_rules": dict(binary_rules),
    "unary_rules": dict(unary_rules),
    "lexical_rules": dict(lexical_rules),

    # --------------------------------------------------------
    # Q4 parser/helper functions
    # --------------------------------------------------------

    "cky_parse": cky_parse,
    "reconcile_q1_tags": reconcile_q1_tags,
    "analyze_sentence": analyze_sentence,

    # --------------------------------------------------------
    # Q4 configuration
    # --------------------------------------------------------

    "MERGE_PROBABILITY": MERGE_PROBABILITY,
    "GRAMMAR_TRIGGER_N": GRAMMAR_TRIGGER_N,
    "Q4_K": Q4_K
}


with open(
    "q4_model_bundle.pkl",
    "wb"
) as f:

    cloudpickle.dump(
        q4_bundle,
        f
    )


print(
    "Saved Q4 model bundle successfully."
)

print(
    "Bundle contains Q1 + Q3 + Q4 models and parser functions."
)

print(
    "q4_model_bundle.pkl"
)

Saved Q4 model bundle successfully.
Bundle contains Q1 + Q3 + Q4 models and parser functions.
q4_model_bundle.pkl


In [58]:
# ============================================================
# Q4 SELF-CHECK  (run this last - every line should say PASS)
# ============================================================

def check(name, condition, detail=""):
    print(("PASS  " if condition else "FAIL  ") + name
          + (f"   {detail}" if detail else ""))
    return bool(condition)

results = []

# 1. SPELLING: a typo must reach the spelling corrector, not the segmenter
for typo, expected in [("sentnce", "sentence"), ("teh", "the"),
                       ("recieve", "receive")]:
    r = resolve_token(typo)
    results.append(check(f"spell   {typo} -> {expected}",
                         r["kind"] == "spell" and r["words"] == [expected],
                         str(r)))

# These must be routed to the SPELLING stage (what they are corrected TO
# depends on Q3's frequency-based ranking, e.g. hav -> had).
for typo in ["hav", "wrold", "gud", "speling"]:
    r = resolve_token(typo)
    results.append(check(f"spell   {typo} is a SPELL event (not a segment)",
                         r["kind"] == "spell", str(r)))

# 2. FAST TYPING: merged words must still be split
for merged, expected in [("thecat", ["the", "cat"]),
                         ("iwould", ["i", "would"]),
                         ("goodfeeling", ["good", "feeling"]),
                         ("satonthe", ["sat", "on", "the"])]:
    r = resolve_token(merged)
    results.append(check(f"segment {merged} -> {' | '.join(expected)}",
                         r["kind"] == "segment" and r["words"] == expected,
                         str(r)))

# 3. Clean words are untouched
for w in ["the", "world", "feeling"]:
    results.append(check(f"keep    {w}", resolve_token(w)["kind"] == "keep"))

# 4. Real-word errors are still caught by the grammar trigger
rw = grammar_trigger_check("i would like to sea the world".split())
results.append(check("real-word  sea -> see flagged",
                     any(a["original"] == "sea" for a in rw["real_word_alerts"]),
                     str([(a["original"], a["suggestion"]) for a in rw["real_word_alerts"]])))

# 5. Bulk: 1,000 injected typos / merges recovered by the NEW pipeline
random.seed(7)
pool = [w for w in vocab if w.isalpha() and len(w) >= 4
        and unigram_freq.get(w, 0) >= 5]
typos = [(w, corrupt_word(w)) for w in random.sample(pool, 1000)]
typos = [(w, t) for w, t in typos if t]
typo_ok = sum(resolve_token(t)["words"] == [w] for w, t in typos)
typo_spell = sum(resolve_token(t)["kind"] == "spell" for w, t in typos)
print(f"\nTypos reaching the spell corrector : {typo_spell}/{len(typos)}")
print(f"Typos corrected exactly            : {typo_ok}/{len(typos)}")
results.append(check("bulk typos reach spelling stage >= 90%",
                     typo_spell / len(typos) >= 0.90))

common = [w for w in vocab if w.isalpha() and unigram_freq.get(w, 0) >= 50
          and w in vocab_en]
pairs = [(random.choice(common), random.choice(common)) for _ in range(1000)]
pairs = [(a, b) for a, b in pairs if a + b not in vocab and a + b not in vocab_en]
merge_ok = sum(resolve_token(a + b)["words"] == [a, b] for a, b in pairs)
print(f"Missed spaces recovered exactly    : {merge_ok}/{len(pairs)}")
results.append(check("bulk merges recovered >= 95%", merge_ok / len(pairs) >= 0.95))

# 6. Simulation really injects and detects both error types
random.seed(11)
sim = simulate_live_typing(passage_sentences, delay=0, p_merge=0.15, p_typo=0.15)
results.append(check("simulation injected typos AND merges",
                     sim["injected"]["typo"] > 0 and sim["injected"]["merge"] > 0))
results.append(check("simulation logged SPELL + SEGMENT alerts",
                     sim["spelling_corrections"] > 0 and sim["segmentation_merges"] > 0))

print("\n" + "=" * 50)
print(f"{sum(results)}/{len(results)} checks passed")
print("=" * 50)


PASS  spell   sentnce -> sentence   {'kind': 'spell', 'words': ['sentence']}
PASS  spell   teh -> the   {'kind': 'spell', 'words': ['the']}
PASS  spell   recieve -> receive   {'kind': 'spell', 'words': ['receive']}
PASS  spell   hav is a SPELL event (not a segment)   {'kind': 'spell', 'words': ['had']}
PASS  spell   wrold is a SPELL event (not a segment)   {'kind': 'spell', 'words': ['would']}
PASS  spell   gud is a SPELL event (not a segment)   {'kind': 'spell', 'words': ['god']}
PASS  spell   speling is a SPELL event (not a segment)   {'kind': 'spell', 'words': ['selling']}
PASS  segment thecat -> the | cat   {'kind': 'segment', 'words': ['the', 'cat']}
PASS  segment iwould -> i | would   {'kind': 'segment', 'words': ['i', 'would']}
PASS  segment goodfeeling -> good | feeling   {'kind': 'segment', 'words': ['good', 'feeling']}
PASS  segment satonthe -> sat | on | the   {'kind': 'segment', 'words': ['sat', 'on', 'the']}
PASS  keep    the
PASS  keep    world
PASS  keep    feeling
PASS 

In [59]:
%%writefile q4_app.py

import os
import re
import time
import math
import random
import statistics

import cloudpickle
import pandas as pd
import streamlit as st
import nltk

from collections import defaultdict


# ============================================================
# SETUP
# ============================================================

for _pkg in ("punkt", "punkt_tab", "gutenberg"):
    nltk.download(_pkg, quiet=True)

from nltk.corpus import gutenberg


# ============================================================
# LOAD Q4 BUNDLE (cached so Streamlit reruns stay fast)
# ============================================================

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(BASE_DIR, "q4_model_bundle.pkl")


@st.cache_resource
def load_bundle():
    with open(MODEL_PATH, "rb") as f:
        return cloudpickle.load(f)


model = load_bundle()

# Q1
lm_en = model["lm_en"]
hmm_en = model["hmm_en"]
vocab_en = model["vocab_en"]

# Q3
vocab = model["vocab"]
unigram_freq = model["unigram_freq"]
bigram_freq = model["bigram_freq"]
unigram_ctx_freq = model["unigram_ctx_freq"]
correct_nonword = model["correct_nonword"]
correct_realword = model["correct_realword"]

# Q4
cky_parse = model["cky_parse"]

MERGE_PROBABILITY = model["MERGE_PROBABILITY"]
GRAMMAR_TRIGGER_N = model["GRAMMAR_TRIGGER_N"]
Q4_K = model["Q4_K"]

TYPO_PROBABILITY = 0.10
REAL_WORD_MIN_LOG_GAIN = 2.0
TRIGRAM_ALERT_THRESHOLD = -12.0
BIGRAM_ALERT_THRESHOLD = -10.0

LETTERS = "abcdefghijklmnopqrstuvwxyz"

Q1_TO_PTB = {
    "NOUN": "NN", "PROPN": "NNP", "VERB": "VB", "AUX": "VB",
    "ADJ": "JJ", "ADV": "RB", "PRON": "PRP", "DET": "DT",
    "ADP": "IN", "NUM": "CD", "CONJ": "CC", "CCONJ": "CC",
    "SCONJ": "IN", "PART": "RP", "INTJ": "UH", "SYM": "SYM",
    "X": "FW",
}

VALID_TWO = {
    "of", "to", "in", "it", "is", "be", "he", "we", "me", "my", "no",
    "so", "do", "go", "up", "us", "on", "or", "an", "as", "at", "by",
    "if", "am", "oh", "ok", "hi", "ma", "pa", "ah", "ha", "eh", "ex",
    "id", "la", "lo", "mr", "ms", "pi", "re", "yo", "ye",
}


def good_piece(word):
    if len(word) == 1:
        return word in {"a", "i"}
    if len(word) == 2:
        return word in VALID_TWO
    return word in vocab_en and word in vocab


def plausible_split(parts):
    return len(parts) > 1 and all(good_piece(p) for p in parts)


def resolve_token(token):
    token = str(token).lower()

    if not token or not token.isalpha():
        return {"kind": "keep", "words": [token] if token else []}

    if token in vocab or token in vocab_en:
        return {"kind": "keep", "words": [token]}

    parts = lm_en.segment(token)

    if plausible_split(parts):
        return {"kind": "segment", "words": parts}

    corrected = correct_nonword(token, method="B")

    if corrected != token:
        return {"kind": "spell", "words": [corrected]}

    return {"kind": "keep", "words": [token]}


_TOKEN_RE = re.compile(r"^([^A-Za-z]*)([A-Za-z]+)([^A-Za-z]*)$")


def split_punct(raw):
    """'"hello,' -> ('"', 'hello', ',')  ;  non-matching tokens pass through."""
    m = _TOKEN_RE.match(raw)
    if m:
        return m.group(1), m.group(2), m.group(3)
    return "", raw, ""


def process_raw_token(raw):
    """Run one typed token through segmentation + spelling.

    Returns clean words (alpha only, used by the LMs), display words
    (punctuation re-attached) and the alert events."""
    lead, core, trail = split_punct(raw)
    res = resolve_token(core)

    events = []
    words = res["words"]

    if res["kind"] == "segment":
        tags = hmm_en.tag(words)
        events.append(
            ("SEGMENT-ALERT",
             f"{core} -> {' | '.join(words)} | POS={tags}")
        )
    elif res["kind"] == "spell":
        events.append(("SPELL-ALERT", f"{core} -> {words[0]}"))

    display = list(words)
    if display:
        display[0] = lead + display[0]
        display[-1] = display[-1] + trail
    elif lead or trail:
        display = [lead + trail]

    clean = [w for w in words if w.isalpha()]
    return {
        "kind": res["kind"],
        "words": words,
        "clean": clean,
        "display": display,
        "events": events,
    }


# ============================================================
# BIGRAM / TRIGRAM
# ============================================================

V = len(vocab) + 2


def q4_bigram_prob(w1, w2):
    return (bigram_freq.get((w1, w2), 0) + Q4_K) / (
        unigram_ctx_freq.get(w1, 0) + Q4_K * V
    )


def sentence_bigram_score(words):
    tokens = ["<s>"] + [str(w).lower() for w in words if str(w).strip()] + ["</s>"]
    return sum(
        math.log(max(q4_bigram_prob(tokens[i], tokens[i + 1]), 1e-12))
        for i in range(len(tokens) - 1)
    )


def sentence_trigram_score(words):
    clean_words = [str(w).lower() for w in words if str(w).strip()]
    if not clean_words:
        return 0.0

    score = 0.0
    pp, p = "<s>", "<s>"
    for w in clean_words + ["</s>"]:
        score += math.log(max(lm_en.word_probability(pp, p, w), 1e-12))
        pp, p = p, w
    return score


def local_bigram_log_score(words, index):
    if index < 0 or index >= len(words):
        return -1e9
    word = words[index]
    prev = words[index - 1] if index > 0 else "<s>"
    nxt = words[index + 1] if index < len(words) - 1 else "</s>"
    return (
        math.log(max(q4_bigram_prob(prev, word), 1e-12))
        + math.log(max(q4_bigram_prob(word, nxt), 1e-12))
    )


def grammar_trigger_check(words):
    window = [str(w).lower() for w in words if str(w).strip()][-GRAMMAR_TRIGGER_N:]

    if len(window) < 2:
        return {"grammar_alert": False, "real_word_alerts": [],
                "avg_bigram": 0.0, "avg_trigram": 0.0}

    b_score = sentence_bigram_score(window)
    t_score = sentence_trigram_score(window)
    denom = max(len(window) + 1, 1)

    real_word_alerts = []
    for i, word in enumerate(window):
        if not word.isalpha() or word not in vocab:
            continue
        try:
            corrected, changed = correct_realword(window, i, method="B")
        except Exception:
            continue
        if not changed or not corrected or corrected == word:
            continue

        cand = list(window)
        cand[i] = corrected
        gain = local_bigram_log_score(cand, i) - local_bigram_log_score(window, i)
        if gain >= REAL_WORD_MIN_LOG_GAIN:
            real_word_alerts.append(
                {"index": i, "original": word,
                 "suggestion": corrected, "log_gain": gain}
            )

    return {
        "grammar_alert": (b_score / denom < BIGRAM_ALERT_THRESHOLD
                          or t_score / denom < TRIGRAM_ALERT_THRESHOLD),
        "real_word_alerts": real_word_alerts,
        "avg_bigram": b_score / denom,
        "avg_trigram": t_score / denom,
    }


def grammar_events(result):
    events = []
    if result["grammar_alert"]:
        events.append((
            "GRAMMAR-ALERT",
            "Implausible recent word sequence "
            f"(bigram avg={result['avg_bigram']:.2f}, "
            f"trigram avg={result['avg_trigram']:.2f})",
        ))
    for a in result["real_word_alerts"]:
        events.append((
            "GRAMMAR-ALERT",
            f"Possible real-word error: {a['original']} -> "
            f"{a['suggestion']} (log gain={a['log_gain']:.2f})",
        ))
    return events


# ============================================================
# FINAL SENTENCE ANALYSIS (PCFG -> trigram -> bigram)
# ============================================================

def analyze_words(words):
    words = [w for w in words if w.isalpha()]
    if not words:
        return None

    ptb = [Q1_TO_PTB.get(t) for t in hmm_en.tag(words)]

    tree, pcfg_score = None, -1e9
    try:
        tree, pcfg_score = cky_parse(words, q1_ptb_tags=ptb)
    except Exception:
        pass

    parsed = tree is not None
    b = sentence_bigram_score(words)
    t = sentence_trigram_score(words)
    t_avg = t / max(len(words) + 1, 1)

    if parsed:
        method, verdict = "PCFG", "Likely grammatical"
    elif t_avg >= TRIGRAM_ALERT_THRESHOLD:
        method, verdict = "Trigram", "Likely grammatical"
    else:
        method, verdict = "Bigram", "Possibly ungrammatical"

    return {
        "pcfg_result": "PARSED" if parsed else "UNPARSEABLE",
        "pcfg_score": pcfg_score,
        "bigram_score": b,
        "trigram_score": t,
        "chosen_method": method,
        "final_verdict": verdict,
    }


# ============================================================
# STREAM ENGINE (shared by Live Typing + Simulated Passage)
# ============================================================

class Stream:
    """Feeds raw typed tokens through segmentation -> spelling and
    fires the grammar check every GRAMMAR_TRIGGER_N words."""

    def __init__(self):
        self.clean = []          # alpha words, fed to the LMs
        self.display = []        # words with punctuation, shown to user
        self.alerts = []         # (type, message)
        self.token_latencies = []
        self.grammar_latencies = []
        self.merges = 0
        self.spells = 0
        self._since_trigger = 0

    def feed(self, raw):
        t0 = time.perf_counter()
        out = process_raw_token(raw)
        self.token_latencies.append((time.perf_counter() - t0) * 1000)

        new_events = list(out["events"])
        for kind, _ in out["events"]:
            if kind == "SEGMENT-ALERT":
                self.merges += 1
            elif kind == "SPELL-ALERT":
                self.spells += 1

        self.clean.extend(out["clean"])
        self.display.extend(out["display"])

        self._since_trigger += len(out["clean"])
        while self._since_trigger >= GRAMMAR_TRIGGER_N:
            self._since_trigger -= GRAMMAR_TRIGGER_N
            g0 = time.perf_counter()
            res = grammar_trigger_check(self.clean[-GRAMMAR_TRIGGER_N:])
            self.grammar_latencies.append((time.perf_counter() - g0) * 1000)
            new_events.extend(grammar_events(res))

        self.alerts.extend(new_events)
        return out, new_events


def show_alert(container, kind, msg):
    text = f"[{kind}] {msg}"
    if kind == "SEGMENT-ALERT":
        container.info(text)
    elif kind == "SPELL-ALERT":
        container.warning(text)
    else:
        container.error(text)


def latency_metrics(stream):
    c1, c2, c3, c4 = st.columns(4)
    if stream.token_latencies:
        c1.metric("Avg segmentation + spelling latency",
                  f"{statistics.mean(stream.token_latencies):.3f} ms")
    if stream.grammar_latencies:
        c2.metric("Avg grammar-trigger latency",
                  f"{statistics.mean(stream.grammar_latencies):.3f} ms")
    c3.metric("Segmentation merges resolved", stream.merges)
    c4.metric("Spelling corrections", stream.spells)


# ============================================================
# SIMULATION HELPERS
# ============================================================

def corrupt_word(word):
    for _ in range(20):
        i = random.randrange(len(word))
        op = random.choice(["delete", "substitute", "transpose", "insert"])
        if op == "delete":
            t = word[:i] + word[i + 1:]
        elif op == "substitute":
            t = word[:i] + random.choice(LETTERS) + word[i + 1:]
        elif op == "transpose" and i < len(word) - 1:
            t = word[:i] + word[i + 1] + word[i] + word[i + 2:]
        else:
            t = word[:i] + random.choice(LETTERS) + word[i:]
        if t != word and t not in vocab and t not in vocab_en:
            return t
    return None


def simulate_typed_stream(tokens, p_merge, p_typo):
    """Return [{typed, expected, kind}] - missing spaces and typos injected."""
    out, i = [], 0
    while i < len(tokens):
        tok = tokens[i]
        low = tok.lower()

        if (i < len(tokens) - 1 and tok.isalpha() and tokens[i + 1].isalpha()
                and random.random() < p_merge):
            nxt = tokens[i + 1].lower()
            if (low + nxt) not in vocab and (low + nxt) not in vocab_en:
                out.append({"typed": low + nxt, "expected": [low, nxt],
                            "kind": "merge"})
                i += 2
                continue

        if (tok.isalpha() and len(tok) >= 4 and low in vocab
                and random.random() < p_typo):
            t = corrupt_word(low)
            if t:
                out.append({"typed": t, "expected": [low], "kind": "typo"})
                i += 1
                continue

        out.append({"typed": low, "expected": [low], "kind": "clean"})
        i += 1
    return out


@st.cache_data(show_spinner=False)
def gutenberg_sentences(file_id):
    raw = gutenberg.raw(file_id)
    return [" ".join(s.split()) for s in nltk.sent_tokenize(raw)]


def sample_random_passage(min_s=5, max_s=8, min_w=5, max_w=25):
    files = list(gutenberg.fileids())
    random.shuffle(files)

    for file_id in files:
        sents = gutenberg_sentences(file_id)
        ok = [min_w <= len(s.split()) <= max_w for s in sents]
        n = random.randint(min_s, max_s)

        starts = [
            i for i in range(len(sents) - n + 1)
            if all(ok[i:i + n])
        ]
        if starts:
            s = random.choice(starts)
            return file_id, sents[s:s + n]

    raise RuntimeError("No suitable Gutenberg passage found.")


# ============================================================
# STREAMLIT UI
# ============================================================

st.set_page_config(page_title="Q4 Background Editor", page_icon="✍️",
                   layout="wide")

st.title("✍️ Integrated Background Editor")
st.caption("Question 1 segmentation + Question 3 spelling correction + "
           "Q4 grammar checking")

mode = st.radio("Mode", ["Live Typing", "Simulated Passage"], horizontal=True)


# ------------------------------------------------------------
# LIVE TYPING
# ------------------------------------------------------------

if mode == "Live Typing":

    text = st.text_area("Type your text:", height=220,
                        placeholder="Start typing...")

    if text.strip():
        stream = Stream()
        rows = []

        for sentence in nltk.sent_tokenize(text):
            m0, s0 = stream.merges, stream.spells
            d0 = len(stream.display)
            c0 = len(stream.clean)

            for raw in sentence.split():
                stream.feed(raw)

            analysis = analyze_words(stream.clean[c0:])
            if analysis:
                analysis["sentence"] = " ".join(stream.display[d0:])
                analysis["segmentation_merges"] = stream.merges - m0
                analysis["spelling_corrections"] = stream.spells - s0
                rows.append(analysis)

        st.subheader("Processed Text")
        st.write(" ".join(stream.display))

        st.subheader("Live Alerts")
        if stream.alerts:
            for kind, msg in stream.alerts:
                show_alert(st, kind, msg)
        else:
            st.success("No alerts detected.")

        st.subheader("Latency")
        latency_metrics(stream)

        st.subheader("Final Sentence Analysis")
        if rows:
            cols = ["sentence", "pcfg_result", "pcfg_score", "bigram_score",
                    "trigram_score", "chosen_method", "final_verdict",
                    "segmentation_merges", "spelling_corrections"]
            st.dataframe(pd.DataFrame(rows)[cols])


# ------------------------------------------------------------
# SIMULATED PASSAGE
# ------------------------------------------------------------

else:

    st.subheader("Simulated Live Passage")
    st.write(
        "Streams a passage word-by-word as if typed fast: some spaces are "
        "dropped (segmentation test) and some words are mistyped "
        "(spelling test). Every token goes through the same pipeline as "
        "Live Typing."
    )

    source = st.radio(
        "Passage source",
        ["Random Gutenberg passage (5-8 sentences)", "My own passage"],
        horizontal=True,
    )

    own_text = ""
    if source == "My own passage":
        own_text = st.text_area("Passage:", height=160)

    c1, c2, c3, c4 = st.columns(4)
    p_merge = c1.slider("Missed-space probability p", 0.0, 0.5,
                        float(MERGE_PROBABILITY), 0.01)
    p_typo = c2.slider("Typo probability", 0.0, 0.5,
                       float(TYPO_PROBABILITY), 0.01)
    delay = c3.slider("Delay per token (s)", 0.0, 0.3, 0.03, 0.01)
    seed_txt = c4.text_input("Random seed (optional)", "")

    if st.button("Run simulation", type="primary"):

        if seed_txt.strip():
            random.seed(seed_txt.strip())

        if source == "My own passage":
            if not own_text.strip():
                st.error("Paste a passage first.")
                st.stop()
            file_id = "user input"
            sentences = nltk.sent_tokenize(own_text)
        else:
            with st.spinner("Sampling a random Gutenberg passage..."):
                file_id, sentences = sample_random_passage()

        stream = Stream()
        rows = []
        injected = {"merge": 0, "typo": 0}
        recovered = {"merge": 0, "typo": 0}

        live = st.empty()
        with live.container():
            st.markdown(f"**Source:** `{file_id}` - {len(sentences)} sentences")
            typed_box = st.empty()
            processed_box = st.empty()
            alert_box = st.container()

            for sentence in sentences:
                m0, s0 = stream.merges, stream.spells
                d0, c0 = len(stream.display), len(stream.clean)

                items = simulate_typed_stream(
                    nltk.word_tokenize(sentence), p_merge, p_typo
                )

                for item in items:
                    typed_box.markdown(
                        f"**Typing:** `{item['typed']}`"
                        + ("" if item["kind"] == "clean"
                           else f"  _(injected {item['kind']})_")
                    )

                    out, events = stream.feed(item["typed"])

                    if item["kind"] in injected:
                        injected[item["kind"]] += 1
                        if out["words"] == item["expected"]:
                            recovered[item["kind"]] += 1

                    processed_box.write(" ".join(stream.display))
                    for kind, msg in events:
                        show_alert(alert_box, kind, msg)

                    if delay:
                        time.sleep(delay)

                analysis = analyze_words(stream.clean[c0:])
                if analysis:
                    analysis["sentence"] = " ".join(stream.display[d0:])
                    analysis["segmentation_merges"] = stream.merges - m0
                    analysis["spelling_corrections"] = stream.spells - s0
                    rows.append(analysis)

        live.empty()

        st.session_state["sim"] = {
            "file_id": file_id,
            "original": sentences,
            "processed": " ".join(stream.display),
            "alerts": stream.alerts,
            "stream": stream,
            "rows": rows,
            "injected": injected,
            "recovered": recovered,
        }

    sim = st.session_state.get("sim")

    if sim:
        st.divider()
        st.subheader("Original passage")
        st.caption(f"Source: {sim['file_id']}")
        for i, s in enumerate(sim["original"], 1):
            st.write(f"{i}. {s}")

        st.subheader("Processed passage")
        st.write(sim["processed"])

        st.subheader("Alerts")
        if sim["alerts"]:
            for kind, msg in sim["alerts"]:
                show_alert(st, kind, msg)
        else:
            st.success("No alerts - nothing was injected or detected.")

        st.subheader("Latency and counts")
        latency_metrics(sim["stream"])

        inj, rec = sim["injected"], sim["recovered"]
        k1, k2 = st.columns(2)
        k1.metric("Missed spaces recovered exactly",
                  f"{rec['merge']} / {inj['merge']}")
        k2.metric("Typos corrected exactly",
                  f"{rec['typo']} / {inj['typo']}")

        st.subheader("Final passage analysis")
        if sim["rows"]:
            cols = ["sentence", "pcfg_result", "pcfg_score", "bigram_score",
                    "trigram_score", "chosen_method", "final_verdict",
                    "segmentation_merges", "spelling_corrections"]
            st.dataframe(pd.DataFrame(sim["rows"])[cols])


Overwriting q4_app.py


In [60]:
!pip install -q streamlit

In [61]:
# ============================================================
# START Q4 STREAMLIT APP - COLAB
# ============================================================

!pkill -f "streamlit run q4_app.py" || true

!python -m streamlit run q4_app.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > streamlit.log 2>&1 &

import time
time.sleep(5)

print("=" * 60)
print("STREAMLIT LOG")
print("=" * 60)

!cat streamlit.log

^C
STREAMLIT LOG


2026-10-11 09:54:20.392 Uvicorn server started on 0.0.0.0:8501


In [62]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>